# Clean v6 feature-engineering notebook — ML1 Task 2 salary prediction

This notebook keeps the winning v6 idea as the foundation and adds only small, course-safe feature-engineering blocks.

**What stays from v6**
- Same train / validation / internal-test split.
- Raw validation and internal-test targets.
- `SVR(kernel="rbf")` with the known strong setting: `C=0.2`, `gamma=0.0007`, `epsilon=0.05`.
- `log_floor500` target transformation.
- `high_mild` sample weights.
- Dense target-encoded and multi-select features from v6.
- No validation multiplier.

**What is added**
- Manual experience bins.
- Missingness-profile counts.
- Compact technology-family counts.
- A few interpretable interactions.
- A small `PolynomialFeatures(interaction_only=True)` block on dense variables only.
- Optional LASSO feature selection variants.

**What is intentionally excluded**
- PCA, PowerTransformer, QuantileTransformer, RobustScaler, SplineTransformer.
- Random Forest, Gradient Boosting, XGBoost, CatBoost.
- Unbounded post-hoc tail uplift or raw-dollar tail specialist.


In [1]:
# ===============================================================
# 1. Imports and global configuration
# ===============================================================
from __future__ import annotations

import json
import math
import os
import pickle
import re
import time
import warnings
from collections import Counter, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.exceptions import ConvergenceWarning
from pandas.errors import PerformanceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=PerformanceWarning)

RANDOM_STATE = 42
TRAIN_PATH = Path("train.csv")
TEST_PATH = Path("test.csv")
SAMPLE_SUBMISSION_PATH = Path("sample_submission.csv")
OUTPUT_DIR = Path("clean_v6_feature_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = "annual.pay.usd"
ID_COL = "id"

# Use FAST_MODE first. After the notebook runs end-to-end, set FAST_MODE=False for a wider search.
FAST_MODE = True

# Keep the internal-test set for final locked evaluation only.
VAL_SIZE = 0.15
INTERNAL_TEST_SIZE = 0.15
N_SPLITS_CV = 5

# Salary thresholds used for tail diagnostics and engineered risk features.
LOW_SALARY_FLOOR = 500.0
HIGH_SALARY_100K = 100_000.0
HIGH_SALARY_200K = 200_000.0
HIGH_SALARY_500K = 500_000.0
LOW_SALARY_10K = 10_000.0

# Safety clipping for final predictions only. This prevents numerical accidents, not metric gaming.
PRED_MIN_USD = 1.0
PRED_MAX_USD = 2_000_000.0

print("Configuration loaded — clean v6 feature-engineering notebook.")
print("FAST_MODE:", FAST_MODE)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


Configuration loaded — clean v6 feature-engineering notebook.
FAST_MODE: True
OUTPUT_DIR: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\clean_v6_feature_outputs


In [2]:
# ===============================================================
# 2. Load data and split
# ===============================================================
def normalize_text_value(value):
    """Normalize text while preserving missing values."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    value = value.replace("’", "'").replace("‘", "'").replace("`", "'")
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].map(normalize_text_value)
    # normalize common "Other" labels
    replacements = {
        "Other:": "Other",
        "Other (please specify):": "Other",
    }
    for col in ["dev.role", "industry", "first.help.source"]:
        if col in df.columns:
            df[col] = df[col].replace(replacements)
    return df


def make_region_strata(region_series: pd.Series, min_count: int = 10) -> pd.Series:
    counts = region_series.value_counts(dropna=False)
    return region_series.where(region_series.map(counts) >= min_count, "__RARE_REGION__")


def summarize_salary_split(name: str, y: pd.Series) -> Dict[str, Any]:
    y = pd.Series(y).astype(float)
    return {
        "split": name,
        "n": int(len(y)),
        "mean": float(y.mean()),
        "median": float(y.median()),
        "min": float(y.min()),
        "max": float(y.max()),
        "lt_500": int((y < LOW_SALARY_FLOOR).sum()),
        "gt_100k": int((y > HIGH_SALARY_100K).sum()),
        "gt_200k": int((y > HIGH_SALARY_200K).sum()),
        "p01": float(y.quantile(0.01)),
        "p05": float(y.quantile(0.05)),
        "p25": float(y.quantile(0.25)),
        "p75": float(y.quantile(0.75)),
        "p95": float(y.quantile(0.95)),
        "p99": float(y.quantile(0.99)),
    }


df_all = normalize_object_columns(pd.read_csv(TRAIN_PATH))
df_kaggle = normalize_object_columns(pd.read_csv(TEST_PATH))

assert TARGET in df_all.columns, f"{TARGET} missing from train.csv"
assert ID_COL in df_kaggle.columns, f"{ID_COL} missing from test.csv"

strata_full = make_region_strata(df_all["region"]) if "region" in df_all.columns else None

df_trainval, df_internal_test = train_test_split(
    df_all,
    test_size=INTERNAL_TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_full,
)

strata_trainval = make_region_strata(df_trainval["region"]) if "region" in df_trainval.columns else None
relative_val_size = VAL_SIZE / (1.0 - INTERNAL_TEST_SIZE)

df_train, df_val = train_test_split(
    df_trainval,
    test_size=relative_val_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=strata_trainval,
)

# Reset indices so later masks align cleanly.
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_internal_test = df_internal_test.reset_index(drop=True)
df_kaggle = df_kaggle.reset_index(drop=True)

y_train_raw = df_train[TARGET].astype(float).reset_index(drop=True)
y_val_raw = df_val[TARGET].astype(float).reset_index(drop=True)
y_internal_test_raw = df_internal_test[TARGET].astype(float).reset_index(drop=True)

split_summary = pd.DataFrame([
    summarize_salary_split("train", y_train_raw),
    summarize_salary_split("validation", y_val_raw),
    summarize_salary_split("internal_test", y_internal_test_raw),
])

print("Raw shapes")
print("train:", df_all.shape)
print("kaggle:", df_kaggle.shape)
print("\nSplit sizes")
print("train:", df_train.shape)
print("validation:", df_val.shape)
print("internal_test:", df_internal_test.shape)
print("kaggle:", df_kaggle.shape)
print("\nSalary split summary")
display(split_summary)

split_summary.to_csv(OUTPUT_DIR / "split_salary_summary.csv", index=False)


Raw shapes
train: (2512, 41)
kaggle: (628, 41)

Split sizes
train: (1758, 41)
validation: (377, 41)
internal_test: (377, 41)
kaggle: (628, 41)

Salary split summary


,split,n,mean,median,min,max,lt_500,gt_100k,gt_200k,p01,p05,p25,p75,p95,p99
0,train,1758,51287.046075,41854.5,11.0,4773360.0,57,143,13,76.26,882.55,16073.5,67898.0,117887.45,178901.9
1,validation,377,46288.106101,38113.0,1.0,384552.0,13,36,3,183.80,756.20,15122.0,65008.0,121496.40,169134.0
2,internal_test,377,45797.676393,39989.0,25.0,911275.0,11,23,2,148.76,809.20,18590.0,60052.0,116545.40,156503.0


In [3]:
# ===============================================================
# 3. Feature engineering configuration
# ===============================================================
MULTI_SEP = ";"

NUMERIC_BASE_COLS = [
    "coding.years.total",
    "coding.years.professional",
    "experience.years",
    "job.satisfaction",
]

# Ordinal maps. These create dense numeric features.
ORDINAL_MAPS = {
    "age.group": {"18-24": 21, "25-34": 29.5, "35-44": 39.5, "45-54": 49.5, "55+": 60},
    "education": {
        "Primary/elementary school": 0,
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": 1,
        "Some college/university study without earning a degree": 2,
        "Associate degree (A.A., A.S., etc.)": 3,
        "Something else": 3,
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)": 4,
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)": 5,
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": 6,
    },
    "company.size": {
        "Just me - I am a freelancer, sole proprietor, etc.": 0,
        "2 to 9 employees": 1,
        "10 to 19 employees": 2,
        "20 to 99 employees": 3,
        "100 to 499 employees": 4,
        "500 to 999 employees": 5,
        "1,000 to 4,999 employees": 6,
        "5,000 to 9,999 employees": 7,
        "10,000 or more employees": 8,
    },
    "tech.purchase.influence": {
        "I have little or no influence": 0,
        "I have some influence": 1,
        "I have a great deal of influence": 2,
    },
    "ai.sentiment": {
        "Very unfavorable": 0,
        "Unfavorable": 1,
        "Indifferent": 2,
        "Unsure": 2,
        "Favorable": 3,
        "Very favorable": 4,
    },
    "ai.trust": {
        "Highly distrust": 0,
        "Somewhat distrust": 1,
        "Neither trust nor distrust": 2,
        "Somewhat trust": 3,
        "Highly trust": 4,
    },
    "ai.complex.rating": {
        "Very poor at handling complex tasks": 0,
        "Bad at handling complex tasks": 1,
        "Neither good or bad at handling complex tasks": 2,
        "Good, but not great at handling complex tasks": 3,
        "Very well at handling complex tasks": 4,
    },
    "daily.search.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
    "daily.answer.time": {
        "Less than 15 minutes a day": 0,
        "15-30 minutes a day": 1,
        "30-60 minutes a day": 2,
        "60-120 minutes a day": 3,
        "Over 120 minutes a day": 4,
    },
}

QUASI_ORDINAL_MAPS = {
    "is.dev.professional": {
        "I am a developer by profession": 1,
        "I am not primarily a developer, but I write code sometimes as part of my work/studies": 0,
    },
    "people.manager": {"People manager": 1, "Individual contributor": 0},
    "uses.ai": {"Yes": 2, "No, but I plan to soon": 1, "No, and I don't plan to": 0},
}

# build.vs.buy is not treated as ordinal here. We target-encode it as a category.
CATEGORICAL_TE_COLS = [
    "region",
    "dev.role",
    "industry",
    "employment.type",
    "work.location",
    "company.size",
    "education",
    "cloud.hosting",
    "people.manager",
    "is.dev.professional",
    "uses.ai",
    "build.vs.buy",
    "first.help.source",
    "ai.job.threat",
]

# Interactions for target encoding. These are dense high-signal features.
INTERACTION_TE_PAIRS = [
    ("region", "dev.role"),
    ("region", "employment.type"),
    ("region", "industry"),
    ("region", "work.location"),
]

MULTI_SELECT_COLS = [
    "prog.languages",
    "databases",
    "cloud.platforms",
    "web.frameworks",
    "other.tech",
    "dev.tools",
    "dev.environments",
    "personal.os",
    "work.os",
    "project.mgmt.tools",
    "comm.tools",
    "ai.search.tools",
    "ai.tools.used",
    "side.coding",
    "how.learned.coding",
]

# Feature profiles control how many sparse technology binaries are added.
# top0_dense has zero technology binaries and relies on counts + salary-score features.
FEATURE_PROFILES = {
    "top0_dense": {"top_n_binary_per_multiselect": 0, "include_basic_ohe": False},
    "top3_dense": {"top_n_binary_per_multiselect": 3, "include_basic_ohe": False},
    "top5_dense": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": False},
    "top5_hybrid_ohe": {"top_n_binary_per_multiselect": 5, "include_basic_ohe": True},
}

# Keep search manageable first.
if FAST_MODE:
    # Small but diverse set: no tech binaries vs current best-style top5 binaries.
    FEATURE_PROFILE_LIST = ["top0_dense", "top5_dense"]
else:
    FEATURE_PROFILE_LIST = list(FEATURE_PROFILES.keys())

print("Feature profiles:", FEATURE_PROFILE_LIST)


Feature profiles: ['top0_dense', 'top5_dense']


In [4]:
# ===============================================================
# 4. Target transformation helper
# ===============================================================
@dataclass
class TargetTransformer:
    variant: str
    scaler_: StandardScaler = field(default_factory=StandardScaler)
    upper_clip_: Optional[float] = None
    lower_clip_: Optional[float] = None

    def _prepare_raw(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = pd.Series(y_raw).astype(float).to_numpy()
        y = np.clip(y, PRED_MIN_USD, None)

        if self.variant.endswith("floor500") or "floor500" in self.variant:
            y = np.clip(y, LOW_SALARY_FLOOR, None)

        if "upper_clip" in self.variant:
            if fit:
                self.upper_clip_ = float(np.quantile(y, 0.995))
            y = np.clip(y, None, self.upper_clip_)

        return y

    def _transform_unscaled(self, y_raw: pd.Series, fit: bool = False) -> np.ndarray:
        y = self._prepare_raw(y_raw, fit=fit)
        if self.variant.startswith("log"):
            return np.log(y)
        if self.variant.startswith("sqrt"):
            return np.sqrt(y)
        if self.variant.startswith("cbrt"):
            return np.cbrt(y)
        if self.variant.startswith("raw"):
            return y
        raise ValueError(f"Unknown target variant: {self.variant}")

    def fit(self, y_raw: pd.Series) -> "TargetTransformer":
        z = self._transform_unscaled(y_raw, fit=True).reshape(-1, 1)
        self.scaler_.fit(z)
        return self

    def transform(self, y_raw: pd.Series) -> np.ndarray:
        z = self._transform_unscaled(y_raw, fit=False).reshape(-1, 1)
        return self.scaler_.transform(z).ravel()

    def inverse_transform_to_usd(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        z = self.scaler_.inverse_transform(np.asarray(y_pred_scaled).reshape(-1, 1)).ravel()
        if self.variant.startswith("log"):
            y = np.exp(z)
        elif self.variant.startswith("sqrt"):
            y = np.square(np.clip(z, 0, None))
        elif self.variant.startswith("cbrt"):
            y = np.power(z, 3)
        elif self.variant.startswith("raw"):
            y = z
        else:
            raise ValueError(f"Unknown target variant: {self.variant}")
        return np.clip(y, PRED_MIN_USD, PRED_MAX_USD)

    def inverse_transform_to_log_scale(self, y_pred_scaled: np.ndarray) -> np.ndarray:
        """Return predicted log salary. Useful for smearing calibration on log variants."""
        y_usd = self.inverse_transform_to_usd(y_pred_scaled)
        return np.log(np.clip(y_usd, PRED_MIN_USD, None))


TARGET_VARIANTS = [
    "log_raw",
    "log_floor500",
    "sqrt_raw",
    "sqrt_floor500",
    "cbrt_raw",
    "cbrt_floor500",
    "raw_scaled",
    "raw_floor500_scaled",
    "raw_upper_clip_scaled",
]
if FAST_MODE:
    # The previous run suggested sqrt/raw targets are strongest; cbrt is added to reduce log compression while remaining less extreme than raw.
    TARGET_VARIANTS = ["sqrt_raw", "cbrt_raw", "raw_scaled", "log_floor500"]

print("Target variants:", TARGET_VARIANTS)


Target variants: ['sqrt_raw', 'cbrt_raw', 'raw_scaled', 'log_floor500']


In [5]:
# ===============================================================
# 5. Dense SVR preprocessor
# ===============================================================
def parse_multiselect_cell(value, sep: str = MULTI_SEP) -> List[str]:
    if pd.isna(value):
        return []
    value = normalize_text_value(value)
    if value == "":
        return []
    return [normalize_text_value(item) for item in str(value).split(sep) if normalize_text_value(item) != ""]


def sanitize_feature_name(name: str) -> str:
    name = normalize_text_value(name)
    name = str(name).lower()
    replacements = {
        " ": "_", "/": "_", ".": "_", "(": "", ")": "", ",": "",
        "'": "", "-": "_", "+": "plus", "#": "sharp", ":": "", "!": "",
        "&": "and", "[": "", "]": "",
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    name = re.sub(r"[^a-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name or "unknown"


def exp_bucket_from_prof_years(series: pd.Series) -> pd.Series:
    x = pd.to_numeric(series, errors="coerce")
    return pd.cut(
        x,
        bins=[-np.inf, 1, 3, 5, 10, 20, np.inf],
        labels=["0_1", "1_3", "3_5", "5_10", "10_20", "20_plus"],
    ).astype(str).replace("nan", "__Missing__")


@dataclass
class SmoothedStats:
    mean_map: Dict[str, float]
    high100_map: Dict[str, float]
    high200_map: Dict[str, float]
    low500_map: Dict[str, float]
    count_map: Dict[str, int]
    global_mean: float
    global_high100: float
    global_high200: float
    global_low500: float


@dataclass
class TailFocusedSVRPreprocessor:
    feature_profile: str = "top5_dense"
    te_smoothing: float = 30.0
    rate_smoothing: float = 50.0
    item_smoothing: float = 20.0
    min_missing_rate: float = 0.10

    numeric_medians_: Dict[str, float] = field(default_factory=dict)
    ordinal_fallbacks_: Dict[str, float] = field(default_factory=dict)
    missing_indicator_cols_: List[str] = field(default_factory=list)
    cat_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    interaction_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_item_stats_: Dict[str, SmoothedStats] = field(default_factory=dict)
    multi_top_items_: Dict[str, List[str]] = field(default_factory=dict)
    feature_cols_: List[str] = field(default_factory=list)
    zero_variance_cols_: List[str] = field(default_factory=list)
    global_log_mean_: float = 0.0
    global_high100_: float = 0.0
    global_high200_: float = 0.0
    global_low500_: float = 0.0

    def fit(self, X_raw: pd.DataFrame, y_raw: pd.Series) -> "TailFocusedSVRPreprocessor":
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        X = X.reset_index(drop=True)

        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        self.global_log_mean_ = float(y_log.mean())
        self.global_high100_ = float((y > HIGH_SALARY_100K).mean())
        self.global_high200_ = float((y > HIGH_SALARY_200K).mean())
        self.global_low500_ = float((y < LOW_SALARY_FLOOR).mean())

        missing_rates = X.isna().mean()
        self.missing_indicator_cols_ = [c for c, r in missing_rates.items() if r >= self.min_missing_rate and r > 0]

        # Numeric and ordinal medians.
        X_num = self._make_numeric_features(X, fit_mode=True)
        self.numeric_medians_ = {
            col: float(X_num[col].median()) if X_num[col].notna().any() else 0.0
            for col in X_num.columns
        }
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                self.ordinal_fallbacks_[col] = float(mapped.median()) if mapped.notna().any() else 0.0

        # Target encoding stats for categorical variables.
        self.cat_stats_ = {}
        for col in CATEGORICAL_TE_COLS:
            if col in X.columns:
                values = self._clean_category_series(X[col])
                self.cat_stats_[col] = self._fit_smoothed_stats(values, y)

        # Add experience bucket as a target-encoded pseudo-category.
        if "coding.years.professional" in X.columns:
            exp_bucket = exp_bucket_from_prof_years(X["coding.years.professional"])
            self.cat_stats_["professional_years_bucket"] = self._fit_smoothed_stats(exp_bucket, y)

        # Interaction target encodings.
        self.interaction_stats_ = {}
        for col_a, col_b in INTERACTION_TE_PAIRS:
            if col_a in X.columns and col_b in X.columns:
                key = f"{col_a}__x__{col_b}"
                values = self._interaction_series(X[col_a], X[col_b])
                self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)
        if "region" in X.columns and "coding.years.professional" in X.columns:
            key = "region__x__professional_years_bucket"
            values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
            self.interaction_stats_[key] = self._fit_smoothed_stats(values, y)

        # Multi-select item stats and top items.
        self.multi_item_stats_ = {}
        self.multi_top_items_ = {}
        for col in MULTI_SELECT_COLS:
            if col not in X.columns:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            item_rows = []
            for i, items in enumerate(parsed):
                for item in items:
                    item_rows.append((item, y.iloc[i]))
            if item_rows:
                item_df = pd.DataFrame(item_rows, columns=["item", "salary"])
                self.multi_item_stats_[col] = self._fit_smoothed_stats(item_df["item"], item_df["salary"])
                counts = Counter([item for items in parsed for item in items])
                self.multi_top_items_[col] = [item for item, _ in counts.most_common(10)]
            else:
                self.multi_item_stats_[col] = self._empty_stats()
                self.multi_top_items_[col] = []

        # Determine final columns on training data.
        X_tmp = self.transform(X_raw, fit_call=True)
        self.zero_variance_cols_ = X_tmp.columns[X_tmp.nunique(dropna=False) <= 1].tolist()
        X_tmp = X_tmp.drop(columns=self.zero_variance_cols_, errors="ignore")
        self.feature_cols_ = X_tmp.columns.tolist()
        return self

    def transform(self, X_raw: pd.DataFrame, fit_call: bool = False) -> pd.DataFrame:
        X = normalize_object_columns(X_raw.copy())
        X = X.drop(columns=[TARGET], errors="ignore")
        X = X.reset_index(drop=True)
        pieces = []

        # Missing flags.
        miss = pd.DataFrame(index=X.index)
        for col in self.missing_indicator_cols_:
            if col in X.columns:
                miss[f"{col}_missing"] = X[col].isna().astype(float)
        if "company.size" in X.columns:
            miss["company_size_dontknow"] = (X["company.size"] == "I don't know").astype(float)
        if not miss.empty:
            pieces.append(miss)

        # Dense numeric features.
        X_num = self._make_numeric_features(X, fit_mode=False)
        for col, med in self.numeric_medians_.items():
            if col not in X_num.columns:
                X_num[col] = med
            else:
                X_num[col] = X_num[col].fillna(med)
        pieces.append(X_num.astype(float))

        # Target encodings for categories.
        te_df = pd.DataFrame(index=X.index)
        for col, stats in self.cat_stats_.items():
            if col == "professional_years_bucket":
                if "coding.years.professional" in X.columns:
                    values = exp_bucket_from_prof_years(X["coding.years.professional"])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                values = self._clean_category_series(X[col]) if col in X.columns else pd.Series("__Missing__", index=X.index)
            safe_col = sanitize_feature_name(col)
            self._apply_stats_to_frame(te_df, safe_col, values, stats)

        # Target encodings for interactions.
        for key, stats in self.interaction_stats_.items():
            if key == "region__x__professional_years_bucket":
                if "region" in X.columns and "coding.years.professional" in X.columns:
                    values = self._interaction_series(X["region"], exp_bucket_from_prof_years(X["coding.years.professional"]))
                else:
                    values = pd.Series("__Missing__", index=X.index)
            else:
                parts = key.split("__x__")
                col_a, col_b = parts[0], parts[1]
                if col_a in X.columns and col_b in X.columns:
                    values = self._interaction_series(X[col_a], X[col_b])
                else:
                    values = pd.Series("__Missing__", index=X.index)
            safe_key = sanitize_feature_name(key)
            self._apply_stats_to_frame(te_df, safe_key, values, stats)

        if not te_df.empty:
            pieces.append(te_df.astype(float))

        # Multi-select salary score features and optional top-N binaries.
        multi_df = pd.DataFrame(index=X.index)
        top_n = int(FEATURE_PROFILES[self.feature_profile]["top_n_binary_per_multiselect"])
        for col in MULTI_SELECT_COLS:
            if col not in X.columns or col not in self.multi_item_stats_:
                continue
            parsed = [parse_multiselect_cell(v) for v in X[col]]
            prefix = sanitize_feature_name(col)
            stats = self.multi_item_stats_[col]
            counts = np.array([len(items) for items in parsed], dtype=float)
            multi_df[f"{prefix}_count"] = counts

            # Salary score summaries.
            mean_scores, max_scores, high100_mean, high100_max, high200_mean, low500_mean, low500_max = [], [], [], [], [], [], []
            for items in parsed:
                if len(items) == 0:
                    item_mean = [stats.global_mean]
                    item_high100 = [stats.global_high100]
                    item_high200 = [stats.global_high200]
                    item_low500 = [stats.global_low500]
                else:
                    item_mean = [stats.mean_map.get(item, stats.global_mean) for item in items]
                    item_high100 = [stats.high100_map.get(item, stats.global_high100) for item in items]
                    item_high200 = [stats.high200_map.get(item, stats.global_high200) for item in items]
                    item_low500 = [stats.low500_map.get(item, stats.global_low500) for item in items]
                mean_scores.append(float(np.mean(item_mean)))
                max_scores.append(float(np.max(item_mean)))
                high100_mean.append(float(np.mean(item_high100)))
                high100_max.append(float(np.max(item_high100)))
                high200_mean.append(float(np.mean(item_high200)))
                low500_mean.append(float(np.mean(item_low500)))
                low500_max.append(float(np.max(item_low500)))
            multi_df[f"{prefix}_salary_score_mean"] = mean_scores
            multi_df[f"{prefix}_salary_score_max"] = max_scores
            multi_df[f"{prefix}_high100_score_mean"] = high100_mean
            multi_df[f"{prefix}_high100_score_max"] = high100_max
            multi_df[f"{prefix}_high200_score_mean"] = high200_mean
            multi_df[f"{prefix}_low500_score_mean"] = low500_mean
            multi_df[f"{prefix}_low500_score_max"] = low500_max

            # Optional sparse top-N binaries. Keep small N only.
            if top_n > 0:
                for item in self.multi_top_items_.get(col, [])[:top_n]:
                    item_col = f"{prefix}__{sanitize_feature_name(item)}"
                    multi_df[item_col] = [float(item in items) for items in parsed]

        if not multi_df.empty:
            pieces.append(multi_df.astype(float))

        # Optional basic one-hot for very small categories only.
        if FEATURE_PROFILES[self.feature_profile].get("include_basic_ohe", False):
            small_ohe_cols = ["employment.type", "work.location", "people.manager"]
            ohe_src = pd.DataFrame(index=X.index)
            for col in small_ohe_cols:
                if col in X.columns:
                    ohe_src[col] = X[col].fillna("__Missing__").astype(str)
            if not ohe_src.empty:
                pieces.append(pd.get_dummies(ohe_src, prefix_sep="__", drop_first=True, dtype=float))

        X_out = pd.concat(pieces, axis=1) if pieces else pd.DataFrame(index=X.index)
        X_out = X_out.loc[:, ~X_out.columns.duplicated()].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(float)

        if fit_call:
            return X_out

        X_out = X_out.drop(columns=self.zero_variance_cols_, errors="ignore")
        # Align to training columns.
        for col in self.feature_cols_:
            if col not in X_out.columns:
                X_out[col] = 0.0
        extra_cols = [c for c in X_out.columns if c not in self.feature_cols_]
        if extra_cols:
            X_out = X_out.drop(columns=extra_cols)
        return X_out[self.feature_cols_].astype(float)

    def _make_numeric_features(self, X: pd.DataFrame, fit_mode: bool = False) -> pd.DataFrame:
        out = pd.DataFrame(index=X.index)
        # Basic numeric.
        for col in NUMERIC_BASE_COLS:
            if col in X.columns:
                out[sanitize_feature_name(col)] = pd.to_numeric(X[col], errors="coerce")
            else:
                out[sanitize_feature_name(col)] = np.nan

        total = pd.to_numeric(X.get("coding.years.total", pd.Series(np.nan, index=X.index)), errors="coerce")
        prof = pd.to_numeric(X.get("coding.years.professional", pd.Series(np.nan, index=X.index)), errors="coerce")
        exp = pd.to_numeric(X.get("experience.years", pd.Series(np.nan, index=X.index)), errors="coerce")

        out["years_before_professional"] = (total - prof).clip(lower=0)

        # Ordinal / quasi-ordinal.
        for col, mapping in {**ORDINAL_MAPS, **QUASI_ORDINAL_MAPS}.items():
            safe = sanitize_feature_name(col)
            if col in X.columns:
                mapped = X[col].replace({"I don't know": np.nan}).map(mapping)
                fallback = self.ordinal_fallbacks_.get(col, np.nan)
                out[safe] = mapped.fillna(fallback)
            else:
                out[safe] = self.ordinal_fallbacks_.get(col, 0.0)

        # Multi-select counts.
        count_cols = {}
        for col in MULTI_SELECT_COLS:
            if col in X.columns:
                counts = pd.Series([len(parse_multiselect_cell(v)) for v in X[col]], index=X.index).astype(float)
            else:
                counts = pd.Series(0.0, index=X.index)
            c_name = f"{sanitize_feature_name(col)}_raw_count"
            count_cols[col] = counts
            out[c_name] = counts
        total_tech_count = sum(count_cols.values()) if count_cols else pd.Series(0.0, index=X.index)
        out["total_multiselect_count"] = total_tech_count

        # Ratios and dense interactions. Add 1 to denominators for stability.
        out["experience_to_professional_ratio"] = (exp / (prof + 1)).clip(lower=0, upper=10)
        age_mid = out.get("age_group", pd.Series(np.nan, index=X.index))
        out["professional_years_to_age_ratio"] = (prof / age_mid.replace(0, np.nan)).clip(lower=0, upper=1)
        out["years_before_professional_ratio"] = (out["years_before_professional"] / (total + 1)).clip(lower=0, upper=1)

        lang_count = count_cols.get("prog.languages", pd.Series(0.0, index=X.index))
        cloud_count = count_cols.get("cloud.platforms", pd.Series(0.0, index=X.index))
        db_count = count_cols.get("databases", pd.Series(0.0, index=X.index))
        ai_count = count_cols.get("ai.tools.used", pd.Series(0.0, index=X.index))
        out["language_count_x_professional_years"] = lang_count * prof.fillna(0)
        out["cloud_count_x_professional_years"] = cloud_count * prof.fillna(0)
        out["database_count_x_professional_years"] = db_count * prof.fillna(0)
        out["ai_count_x_professional_years"] = ai_count * prof.fillna(0)
        out["total_tech_count_x_professional_years"] = total_tech_count * prof.fillna(0)

        manager = out.get("people_manager", pd.Series(0.0, index=X.index)).fillna(0)
        company = out.get("company_size", pd.Series(0.0, index=X.index)).fillna(0)
        influence = out.get("tech_purchase_influence", pd.Series(0.0, index=X.index)).fillna(0)
        out["manager_x_company_size"] = manager * company
        out["influence_x_company_size"] = influence * company
        out["large_company_flag"] = (company >= 6).astype(float)
        out["senior_professional_10y"] = (prof >= 10).astype(float)
        out["senior_professional_20y"] = (prof >= 20).astype(float)
        out["high_experience_10y"] = (exp >= 10).astype(float)
        out["high_experience_20y"] = (exp >= 20).astype(float)

        if "employment.type" in X.columns:
            emp = X["employment.type"].fillna("__Missing__").astype(str)
            out["is_full_time"] = (emp == "Full-time").astype(float)
            out["is_freelance"] = emp.str.contains("Freelance", case=False, na=False).astype(float)
            out["is_student"] = emp.str.contains("Student", case=False, na=False).astype(float)
            out["is_part_time"] = emp.str.contains("Part-time", case=False, na=False).astype(float)
        if "work.location" in X.columns:
            wl = X["work.location"].fillna("__Missing__").astype(str)
            out["is_remote"] = (wl == "Remote").astype(float)
            out["is_hybrid"] = wl.str.contains("Hybrid", case=False, na=False).astype(float)

        # Tail-oriented dense interactions. These are still derived only from available predictors.
        out["senior_x_large_company"] = out["senior_professional_10y"] * out["large_company_flag"]
        out["senior20_x_large_company"] = out["senior_professional_20y"] * out["large_company_flag"]
        out["manager_x_senior"] = manager * out["senior_professional_10y"]
        out["manager_x_senior20"] = manager * out["senior_professional_20y"]
        out["remote_x_senior"] = out.get("is_remote", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["fulltime_x_senior"] = out.get("is_full_time", pd.Series(0.0, index=X.index)) * out["senior_professional_10y"]
        out["freelance_x_low_experience"] = out.get("is_freelance", pd.Series(0.0, index=X.index)) * (prof.fillna(0) <= 1).astype(float)
        out["student_or_parttime_low_salary_risk"] = (out.get("is_student", pd.Series(0.0, index=X.index)) + out.get("is_part_time", pd.Series(0.0, index=X.index))).clip(upper=1)

        return out

    def _clean_category_series(self, s: pd.Series) -> pd.Series:
        return s.fillna("__Missing__").astype(str).map(normalize_text_value).fillna("__Missing__")

    def _interaction_series(self, a: pd.Series, b: pd.Series) -> pd.Series:
        a = self._clean_category_series(a)
        b = self._clean_category_series(b)
        return a.astype(str) + "__x__" + b.astype(str)

    def _fit_smoothed_stats(self, values: pd.Series, y_raw: pd.Series) -> SmoothedStats:
        values = pd.Series(values).fillna("__Missing__").astype(str).reset_index(drop=True)
        y = pd.Series(y_raw).astype(float).reset_index(drop=True)
        y_log = np.log(np.clip(y, PRED_MIN_USD, None))
        df = pd.DataFrame({"value": values, "y": y, "log_y": y_log})
        grouped = df.groupby("value")
        counts = grouped.size()
        mean_log = grouped["log_y"].mean()
        high100 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_100K).mean()))
        high200 = grouped["y"].apply(lambda x: float((x > HIGH_SALARY_200K).mean()))
        low500 = grouped["y"].apply(lambda x: float((x < LOW_SALARY_FLOOR).mean()))

        global_mean = float(y_log.mean())
        global_high100 = float((y > HIGH_SALARY_100K).mean())
        global_high200 = float((y > HIGH_SALARY_200K).mean())
        global_low500 = float((y < LOW_SALARY_FLOOR).mean())

        mean_smooth = (mean_log * counts + global_mean * self.te_smoothing) / (counts + self.te_smoothing)
        high100_smooth = (high100 * counts + global_high100 * self.rate_smoothing) / (counts + self.rate_smoothing)
        high200_smooth = (high200 * counts + global_high200 * self.rate_smoothing) / (counts + self.rate_smoothing)
        low500_smooth = (low500 * counts + global_low500 * self.rate_smoothing) / (counts + self.rate_smoothing)

        return SmoothedStats(
            mean_map=mean_smooth.to_dict(),
            high100_map=high100_smooth.to_dict(),
            high200_map=high200_smooth.to_dict(),
            low500_map=low500_smooth.to_dict(),
            count_map=counts.astype(int).to_dict(),
            global_mean=global_mean,
            global_high100=global_high100,
            global_high200=global_high200,
            global_low500=global_low500,
        )

    def _empty_stats(self) -> SmoothedStats:
        return SmoothedStats({}, {}, {}, {}, {}, self.global_log_mean_, self.global_high100_, self.global_high200_, self.global_low500_)

    def _apply_stats_to_frame(self, out: pd.DataFrame, prefix: str, values: pd.Series, stats: SmoothedStats) -> None:
        values = pd.Series(values).fillna("__Missing__").astype(str)
        out[f"{prefix}_te_log_mean"] = values.map(stats.mean_map).fillna(stats.global_mean).astype(float)
        out[f"{prefix}_te_high100_rate"] = values.map(stats.high100_map).fillna(stats.global_high100).astype(float)
        out[f"{prefix}_te_high200_rate"] = values.map(stats.high200_map).fillna(stats.global_high200).astype(float)
        out[f"{prefix}_te_low500_rate"] = values.map(stats.low500_map).fillna(stats.global_low500).astype(float)
        out[f"{prefix}_te_log_count"] = np.log1p(values.map(stats.count_map).fillna(0).astype(float))

print("TailFocusedSVRPreprocessor ready.")


TailFocusedSVRPreprocessor ready.


In [6]:

# ===============================================================
# 6. Clean extra feature blocks
# ===============================================================
# These blocks are intentionally small and interpretable.
# They build on the v6 feature matrix instead of replacing it.
# No PCA, no trees, no boosting, no outside-ML1 algorithms.

from sklearn.preprocessing import PolynomialFeatures

MISSINGNESS_GROUPS = {
    "work": [
        "employment.type", "work.location", "dev.role", "company.size", "people.manager",
        "industry", "tech.purchase.influence", "cloud.hosting", "job.satisfaction"
    ],
    "tech": [
        "prog.languages", "databases", "cloud.platforms", "web.frameworks",
        "other.tech", "dev.tools", "dev.environments"
    ],
    "ai": [
        "ai.search.tools", "ai.tools.used", "uses.ai", "ai.sentiment", "ai.trust",
        "ai.complex.rating", "ai.job.threat", "daily.search.time", "daily.answer.time"
    ],
    "profile": [
        "region", "age.group", "education", "is.dev.professional",
        "coding.years.total", "coding.years.professional", "experience.years"
    ],
}

# Technology families are deliberately compact. They summarize sparse multi-selects.
TECH_FAMILIES = {
    "data_ml": [
        "Python", "R", "Scala", "SQL", "Snowflake", "BigQuery", "Databricks SQL",
        "Apache Spark", "Hadoop", "Pandas", "NumPy", "Scikit-Learn", "TensorFlow",
        "Torch/PyTorch", "Keras", "MLflow"
    ],
    "cloud_backend": [
        "Amazon Web Services (AWS)", "Microsoft Azure", "Google Cloud", "Cloudflare",
        "OpenShift", "Kubernetes", "Docker", "Terraform", "Ansible", "Java", "Go",
        "Rust", "Kotlin", "PostgreSQL", "Redis", "RabbitMQ", "Apache Kafka", "Kafka"
    ],
    "mobile": [
        "Swift", "Objective-C", "SwiftUI", "Kotlin", "Dart", "Flutter", "React Native",
        "Android Studio"
    ],
    "microsoft_enterprise": [
        "C#", ".NET Framework", ".NET", "ASP.NET", "ASP.NET Core", "Microsoft Azure",
        "Microsoft SQL Server", "PowerShell", "Visual Studio", "Visual Studio Code"
    ],
    "frontend": [
        "HTML/CSS", "JavaScript", "TypeScript", "React", "Angular", "AngularJS",
        "Vue.js", "Next.js", "Node.js", "jQuery", "Vite", "Webpack", "Yarn", "npm"
    ],
    "devops": [
        "Docker", "Kubernetes", "Ansible", "Terraform", "OpenShift", "Puppet",
        "Chef", "Make", "CMake", "Gradle", "Maven", "Git", "Jira"
    ],
    "rare_functional": [
        "Elixir", "Erlang", "F#", "Haskell", "Clojure", "Lisp", "OCaml", "Scala"
    ],
}


def _safe_numeric(series: pd.Series, fallback: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(fallback).astype(float)


def _col(df: pd.DataFrame, name: str, default: float = 0.0) -> pd.Series:
    if name in df.columns:
        return pd.to_numeric(df[name], errors="coerce").fillna(default).astype(float)
    return pd.Series(default, index=df.index, dtype=float)


def _missing_count(raw_df: pd.DataFrame, cols: List[str]) -> pd.Series:
    existing = [c for c in cols if c in raw_df.columns]
    if not existing:
        return pd.Series(0, index=raw_df.index, dtype=float)
    return raw_df[existing].isna().sum(axis=1).astype(float)


def _family_count_for_row(row: pd.Series, family_terms_sanitized: set) -> int:
    count = 0
    for col in MULTI_SELECT_COLS:
        if col not in row.index:
            continue
        items = [sanitize_feature_name(x) for x in parse_multiselect_cell(row[col])]
        count += sum(1 for item in items if item in family_terms_sanitized)
    return int(count)


@dataclass
class CleanExtraFeatureEngineer:
    """Adds only the controlled, ML1-safe feature blocks selected in this notebook."""
    blocks: Tuple[str, ...] = ()
    polynomial_degree: int = 2
    poly_: Optional[PolynomialFeatures] = None
    poly_cols_: List[str] = field(default_factory=list)
    poly_feature_names_: List[str] = field(default_factory=list)
    columns_: List[str] = field(default_factory=list)

    def _add_nonpoly_features(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X = X_base.copy()
        raw = raw_df.reset_index(drop=True).copy()
        X = X.reset_index(drop=True)

        # A. Manual experience bins
        if "experience_bins" in self.blocks:
            prof_years = _col(X, "coding_years_professional", 0.0)
            bins = [-np.inf, 1, 3, 6, 10, 15, 20, np.inf]
            labels = ["0_1", "2_3", "4_6", "7_10", "11_15", "16_20", "21plus"]
            exp_bin = pd.cut(prof_years, bins=bins, labels=labels)
            for lab in labels:
                X[f"clean_expbin_{lab}"] = (exp_bin.astype(str) == lab).astype(float)
            X["clean_expbin_11plus"] = (prof_years >= 11).astype(float)
            X["clean_expbin_16plus"] = (prof_years >= 16).astype(float)

        # B. Missingness profile features
        if "missingness" in self.blocks:
            group_counts = {}
            for group_name, cols in MISSINGNESS_GROUPS.items():
                cnt = _missing_count(raw, cols)
                group_counts[group_name] = cnt
                X[f"clean_missing_{group_name}_count"] = cnt
            total = sum(group_counts.values()) if group_counts else pd.Series(0, index=X.index)
            X["clean_missing_total_count"] = total.astype(float)
            X["clean_has_complete_work_profile"] = (group_counts.get("work", pd.Series(0, index=X.index)) == 0).astype(float)
            X["clean_has_no_tech_stack"] = (group_counts.get("tech", pd.Series(0, index=X.index)) >= 4).astype(float)

        # C. Compact technology-family counts
        if "tech_families" in self.blocks:
            for fam, terms in TECH_FAMILIES.items():
                terms_sanitized = {sanitize_feature_name(t) for t in terms}
                X[f"clean_techfam_{fam}_count"] = raw.apply(lambda row: _family_count_for_row(row, terms_sanitized), axis=1).astype(float)
                X[f"clean_techfam_{fam}_flag"] = (X[f"clean_techfam_{fam}_count"] > 0).astype(float)
            fam_count_cols = [c for c in X.columns if c.startswith("clean_techfam_") and c.endswith("_count")]
            X["clean_techfam_total_count"] = X[fam_count_cols].sum(axis=1) if fam_count_cols else 0.0

        # D. Selective interaction features
        if "interactions" in self.blocks:
            senior = _col(X, "senior_professional_10y", 0.0)
            senior20 = _col(X, "senior_professional_20y", 0.0)
            manager = _col(X, "people_manager", 0.0)
            remote = _col(X, "is_remote", 0.0)
            prof_years = _col(X, "coding_years_professional", 0.0)
            region_log = _col(X, "region_te_log_mean", 0.0)
            region_high = _col(X, "region_te_high100_rate", 0.0)
            role_high = _col(X, "dev_role_te_high100_rate", 0.0)
            data_ml = _col(X, "clean_techfam_data_ml_count", 0.0)
            cloud_backend = _col(X, "clean_techfam_cloud_backend_count", 0.0)
            devops = _col(X, "clean_techfam_devops_count", 0.0)

            X["clean_senior_x_remote"] = senior * remote
            X["clean_senior_x_manager"] = senior * manager
            X["clean_senior20_x_manager"] = senior20 * manager
            X["clean_senior_x_region_high100"] = senior * region_high
            X["clean_senior_x_role_high100"] = senior * role_high
            X["clean_remote_x_region_high100"] = remote * region_high
            X["clean_profyears_x_region_log_mean"] = prof_years * region_log
            X["clean_profyears_x_data_ml_count"] = prof_years * data_ml
            X["clean_profyears_x_cloud_backend_count"] = prof_years * cloud_backend
            X["clean_profyears_x_devops_count"] = prof_years * devops

        # Clean possible infinities from division/interactions
        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        return X

    def fit(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> "CleanExtraFeatureEngineer":
        X = self._add_nonpoly_features(X_base, raw_df)

        if "poly_small" in self.blocks:
            candidate_cols = [
                "coding_years_professional",
                "experience_years",
                "company_size",
                "total_multiselect_count",
                "people_manager",
                "is_remote",
                "region_te_log_mean",
                "dev_role_te_high100_rate",
                "clean_missing_total_count",
                "clean_techfam_data_ml_count",
                "clean_techfam_cloud_backend_count",
                "clean_techfam_devops_count",
            ]
            self.poly_cols_ = [c for c in candidate_cols if c in X.columns]
            self.poly_ = PolynomialFeatures(
                degree=self.polynomial_degree,
                interaction_only=True,
                include_bias=False
            )
            if len(self.poly_cols_) >= 2:
                self.poly_.fit(X[self.poly_cols_])
                raw_names = list(self.poly_.get_feature_names_out(self.poly_cols_))
                # Keep interaction terms only; original columns already exist in X.
                self.poly_feature_names_ = [name for name in raw_names if " " in name]
            else:
                self.poly_ = None
                self.poly_feature_names_ = []

        X_final = self.transform(X_base, raw_df)
        self.columns_ = list(X_final.columns)
        return self

    def transform(self, X_base: pd.DataFrame, raw_df: pd.DataFrame) -> pd.DataFrame:
        X = self._add_nonpoly_features(X_base, raw_df)

        if "poly_small" in self.blocks and self.poly_ is not None and len(self.poly_cols_) >= 2:
            poly_arr = self.poly_.transform(X[self.poly_cols_])
            all_poly_names = list(self.poly_.get_feature_names_out(self.poly_cols_))
            poly_df = pd.DataFrame(poly_arr, columns=all_poly_names, index=X.index)
            for name in self.poly_feature_names_:
                safe_name = "clean_poly_" + sanitize_feature_name(name)
                X[safe_name] = poly_df[name].astype(float)

        X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

        # Align to training columns if fitted.
        if self.columns_:
            for col in self.columns_:
                if col not in X.columns:
                    X[col] = 0.0
            X = X[self.columns_]
        return X

print("Clean extra feature engineer ready.")


Clean extra feature engineer ready.


In [7]:
# ===============================================================
# 6. Metrics, train-only calibration, and sample weights
# ===============================================================
def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def salary_bucket_summary(y_true: pd.Series, y_pred: np.ndarray) -> pd.DataFrame:
    df = pd.DataFrame({"actual": np.asarray(y_true, dtype=float), "pred": np.asarray(y_pred, dtype=float)})
    df["error"] = df["pred"] - df["actual"]
    df["sq_error"] = df["error"] ** 2
    df["abs_error"] = df["error"].abs()
    df["bucket"] = pd.cut(
        df["actual"],
        bins=[0, 500, 10_000, 30_000, 60_000, 100_000, 200_000, np.inf],
        labels=["<500", "500-10k", "10k-30k", "30k-60k", "60k-100k", "100k-200k", "200k+"],
        include_lowest=True,
    )
    out = df.groupby("bucket", observed=False).agg(
        n=("actual", "size"),
        mean_actual=("actual", "mean"),
        mean_pred=("pred", "mean"),
        rmse=("error", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        mae=("abs_error", "mean"),
        mse_sum=("sq_error", "sum"),
    ).reset_index()
    out["mse_contribution_pct"] = 100 * out["mse_sum"] / df["sq_error"].sum()
    return out


# -------------------------------
# Sample-weight schemes
# -------------------------------
WEIGHT_SCHEMES = {
    "none": {"lt10k": 1.0, "gt100k": 1.0, "gt200k": 1.0, "gt500k": 1.0},
    "high_mild": {"lt10k": 1.0, "gt100k": 2.0, "gt200k": 4.0, "gt500k": 6.0},
    "high_medium": {"lt10k": 1.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_balanced": {"lt10k": 2.0, "gt100k": 3.0, "gt200k": 7.0, "gt500k": 12.0},
    "tail_strong": {"lt10k": 2.5, "gt100k": 4.0, "gt200k": 10.0, "gt500k": 18.0},
}


def make_sample_weights(y_raw: pd.Series, scheme: str) -> np.ndarray:
    y = pd.Series(y_raw).astype(float).to_numpy()
    cfg = WEIGHT_SCHEMES.get(scheme)
    if cfg is None:
        raise ValueError(f"Unknown weight scheme: {scheme}")
    w = np.ones(len(y), dtype=float)
    w[y < LOW_SALARY_10K] = cfg["lt10k"]
    w[y > HIGH_SALARY_100K] = cfg["gt100k"]
    w[y > HIGH_SALARY_200K] = cfg["gt200k"]
    w[y > HIGH_SALARY_500K] = cfg["gt500k"]
    # Normalize mean weight to 1 so C remains comparable across schemes.
    w = w / np.mean(w)
    return w


def sample_weight_summary(y_raw: pd.Series, scheme: str) -> Dict[str, float]:
    w = make_sample_weights(y_raw, scheme)
    return {
        "weight_scheme": scheme,
        "weight_mean": float(np.mean(w)),
        "weight_min": float(np.min(w)),
        "weight_max": float(np.max(w)),
        "n_weight_gt1": int((w > 1.01).sum()),
    }


# -------------------------------
# Train-only calibration
# -------------------------------
def fit_train_global_ratio(pred_train_usd: np.ndarray, y_train_raw: pd.Series, cap_low: float = 0.65, cap_high: float = 2.25) -> float:
    pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
    true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
    ratios = np.clip(true / pred, cap_low, cap_high)
    return float(np.mean(ratios))


@dataclass
class BinnedRatioCalibrator:
    n_bins: int = 6
    min_bin_size: int = 25
    smoothing: float = 40.0
    cap_low: float = 0.60
    cap_high: float = 2.75
    global_factor_: float = 1.0
    bin_edges_: Optional[np.ndarray] = None
    bin_factors_: Optional[np.ndarray] = None

    def fit(self, pred_train_usd: np.ndarray, y_train_raw: pd.Series) -> "BinnedRatioCalibrator":
        pred = np.clip(np.asarray(pred_train_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        true = np.clip(pd.Series(y_train_raw).astype(float).to_numpy(), PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        ratios = np.clip(true / pred, self.cap_low, self.cap_high)
        self.global_factor_ = float(np.mean(ratios))
        edges = np.unique(np.quantile(pred_log, np.linspace(0, 1, self.n_bins + 1)))
        if len(edges) <= 2:
            self.bin_edges_ = np.array([-np.inf, np.inf])
            self.bin_factors_ = np.array([self.global_factor_])
            return self
        edges[0] = -np.inf
        edges[-1] = np.inf
        self.bin_edges_ = edges
        bin_ids = np.digitize(pred_log, edges[1:-1], right=True)
        factors = []
        for b in range(len(edges) - 1):
            mask = bin_ids == b
            n = int(mask.sum())
            if n < self.min_bin_size:
                factors.append(self.global_factor_)
            else:
                raw_factor = float(np.mean(ratios[mask]))
                smoothed = (raw_factor * n + self.global_factor_ * self.smoothing) / (n + self.smoothing)
                factors.append(float(np.clip(smoothed, self.cap_low, self.cap_high)))
        self.bin_factors_ = np.array(factors)
        return self

    def apply(self, pred_usd: np.ndarray) -> np.ndarray:
        pred = np.clip(np.asarray(pred_usd, dtype=float), PRED_MIN_USD, PRED_MAX_USD)
        if self.bin_edges_ is None or self.bin_factors_ is None:
            return np.clip(pred * self.global_factor_, PRED_MIN_USD, PRED_MAX_USD)
        pred_log = np.log(pred)
        bin_ids = np.digitize(pred_log, self.bin_edges_[1:-1], right=True)
        factors = self.bin_factors_[bin_ids]
        return np.clip(pred * factors, PRED_MIN_USD, PRED_MAX_USD)


def evaluate_calibrations(
    target_transformer: TargetTransformer,
    pred_train_scaled: np.ndarray,
    pred_val_scaled: np.ndarray,
    y_train_raw_used: pd.Series,
    y_val_raw_eval: pd.Series,
) -> Dict[str, Any]:
    pred_train_usd = target_transformer.inverse_transform_to_usd(pred_train_scaled)
    pred_val_usd_none = target_transformer.inverse_transform_to_usd(pred_val_scaled)

    records = []
    records.append({
        "calibration": "none",
        "factor_or_info": "direct",
        "pred_val_usd": pred_val_usd_none,
        "val_rmse": rmse(y_val_raw_eval, pred_val_usd_none),
    })

    # Global train-only ratio calibration. Unlike validation_multiplier, this is fitted only on training residuals.
    global_ratio = fit_train_global_ratio(pred_train_usd, y_train_raw_used)
    pred_global = np.clip(pred_val_usd_none * global_ratio, PRED_MIN_USD, PRED_MAX_USD)
    records.append({
        "calibration": "train_global_ratio",
        "factor_or_info": global_ratio,
        "pred_val_usd": pred_global,
        "val_rmse": rmse(y_val_raw_eval, pred_global),
    })

    # Binned train-only ratio calibration, designed to correct underprediction differently across prediction ranges.
    binned = BinnedRatioCalibrator(n_bins=6).fit(pred_train_usd, y_train_raw_used)
    pred_binned = binned.apply(pred_val_usd_none)
    records.append({
        "calibration": "train_binned_ratio",
        "factor_or_info": {
            "global_factor": binned.global_factor_,
            "bin_factors": binned.bin_factors_.tolist() if binned.bin_factors_ is not None else None,
        },
        "pred_val_usd": pred_binned,
        "val_rmse": rmse(y_val_raw_eval, pred_binned),
        "calibrator": binned,
    })

    best = min(records, key=lambda r: r["val_rmse"])
    return {"records": records, "best": best}

print("Metrics, sample weights, and train-only calibration helpers ready.")


Metrics, sample weights, and train-only calibration helpers ready.


In [8]:

# ===============================================================
# 7. Modeling helpers — clean ablation over feature blocks
# ===============================================================
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold

BASE_FEATURE_PROFILE = "top5_dense"
BASE_TARGET_VARIANT = "log_floor500"
BASE_WEIGHT_SCHEME = "high_mild"
BASE_SVR_PARAMS = {"C": 0.2, "gamma": 0.0007, "epsilon": 0.05}

def prepare_features(
    train_df_fit: pd.DataFrame,
    val_df_eval: pd.DataFrame,
    kaggle_df_optional: Optional[pd.DataFrame],
    feature_blocks: Tuple[str, ...],
    feature_profile: str = BASE_FEATURE_PROFILE,
):
    y_train_fit_raw = train_df_fit[TARGET].astype(float).reset_index(drop=True)

    base_pre = TailFocusedSVRPreprocessor(feature_profile=feature_profile)
    base_pre.fit(train_df_fit.drop(columns=[TARGET]), y_train_fit_raw)

    X_train_base = base_pre.transform(train_df_fit.drop(columns=[TARGET]))
    X_val_base = base_pre.transform(val_df_eval.drop(columns=[TARGET]))
    X_kaggle_base = base_pre.transform(kaggle_df_optional) if kaggle_df_optional is not None else None

    extra = CleanExtraFeatureEngineer(blocks=feature_blocks)
    extra.fit(X_train_base, train_df_fit.drop(columns=[TARGET]))
    X_train = extra.transform(X_train_base, train_df_fit.drop(columns=[TARGET]))
    X_val = extra.transform(X_val_base, val_df_eval.drop(columns=[TARGET]))
    X_kaggle = extra.transform(X_kaggle_base, kaggle_df_optional) if kaggle_df_optional is not None else None

    scaler_x = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler_x.fit_transform(X_train), columns=X_train.columns)
    X_val_scaled = pd.DataFrame(scaler_x.transform(X_val), columns=X_train.columns)
    X_kaggle_scaled = pd.DataFrame(scaler_x.transform(X_kaggle), columns=X_train.columns) if X_kaggle is not None else None

    target_tf = TargetTransformer(BASE_TARGET_VARIANT).fit(y_train_fit_raw)
    y_train_scaled = pd.Series(target_tf.transform(y_train_fit_raw))

    return {
        "base_preprocessor": base_pre,
        "extra_engineer": extra,
        "scaler_x": scaler_x,
        "target_tf": target_tf,
        "X_train": X_train_scaled,
        "X_val": X_val_scaled,
        "X_kaggle": X_kaggle_scaled,
        "y_train_scaled": y_train_scaled,
        "y_train_raw_used": y_train_fit_raw,
        "feature_names": list(X_train.columns),
    }


def apply_lasso_selection(X_train, y_train_scaled, X_val, X_kaggle=None, top_n: Optional[int] = None):
    """Optional course-safe feature selection: LASSO coefficients select the strongest features."""
    if top_n is None or top_n >= X_train.shape[1]:
        return X_train, X_val, X_kaggle, list(X_train.columns), None

    alpha_grid = {"alpha": np.logspace(-4, -1, 7)}
    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    search = GridSearchCV(
        Lasso(max_iter=20000, random_state=RANDOM_STATE),
        param_grid=alpha_grid,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1,
    )
    search.fit(X_train, y_train_scaled)
    coef = pd.Series(np.abs(search.best_estimator_.coef_), index=X_train.columns)
    selected_cols = coef.sort_values(ascending=False).head(top_n).index.tolist()

    return (
        X_train[selected_cols],
        X_val[selected_cols],
        X_kaggle[selected_cols] if X_kaggle is not None else None,
        selected_cols,
        search.best_params_,
    )


def fit_predict_svr(prepared, lasso_top_n: Optional[int] = None):
    X_train, X_val, X_kaggle, selected_cols, lasso_params = apply_lasso_selection(
        prepared["X_train"],
        prepared["y_train_scaled"],
        prepared["X_val"],
        prepared["X_kaggle"],
        top_n=lasso_top_n,
    )
    model = SVR(kernel="rbf", cache_size=1000, **BASE_SVR_PARAMS)
    sample_weight = make_sample_weights(prepared["y_train_raw_used"], BASE_WEIGHT_SCHEME)
    model.fit(X_train, prepared["y_train_scaled"], sample_weight=sample_weight)

    pred_val_scaled = model.predict(X_val)
    pred_val = prepared["target_tf"].inverse_transform_to_usd(pred_val_scaled)

    pred_kaggle = None
    if X_kaggle is not None:
        pred_kaggle_scaled = model.predict(X_kaggle)
        pred_kaggle = prepared["target_tf"].inverse_transform_to_usd(pred_kaggle_scaled)

    return model, pred_val, pred_kaggle, selected_cols, lasso_params


def prediction_distribution(pred: np.ndarray) -> Dict[str, float]:
    pred = np.asarray(pred, dtype=float)
    return {
        "pred_mean": float(np.mean(pred)),
        "pred_median": float(np.median(pred)),
        "pred_p95": float(np.quantile(pred, 0.95)),
        "pred_p99": float(np.quantile(pred, 0.99)),
        "pred_max": float(np.max(pred)),
        "n_pred_gt100k": int((pred > 100_000).sum()),
        "n_pred_gt150k": int((pred > 150_000).sum()),
        "n_pred_gt200k": int((pred > 200_000).sum()),
    }


def evaluate_feature_block_candidate(name: str, feature_blocks: Tuple[str, ...], lasso_top_n: Optional[int] = None):
    start = time.time()
    prepared = prepare_features(
        train_df_fit=df_train,
        val_df_eval=df_val,
        kaggle_df_optional=df_kaggle,
        feature_blocks=feature_blocks,
    )
    model, pred_val, pred_kaggle, selected_cols, lasso_params = fit_predict_svr(prepared, lasso_top_n=lasso_top_n)

    record = {
        "candidate": name,
        "feature_blocks": ",".join(feature_blocks) if feature_blocks else "base_v6",
        "lasso_top_n": lasso_top_n if lasso_top_n is not None else "none",
        "lasso_best_params": json.dumps(lasso_params) if lasso_params is not None else "",
        "n_features_before_selection": prepared["X_train"].shape[1],
        "n_features_used": len(selected_cols),
        "val_rmse": rmse(y_val_raw, pred_val),
        "val_mae": float(mean_absolute_error(y_val_raw, pred_val)),
        "fit_seconds": float(time.time() - start),
        **prediction_distribution(pred_val),
    }

    if pred_kaggle is not None:
        kg = prediction_distribution(pred_kaggle)
        record.update({f"kaggle_{k}": v for k, v in kg.items()})

    return record, pred_val, pred_kaggle, prepared, model

print("Modeling helpers ready.")


Modeling helpers ready.


In [9]:

# ===============================================================
# 8. Main ablation — keep it small and interpretable
# ===============================================================
# Each row adds one controlled feature block to the exact v6 foundation.
# If a block does not improve validation/internal diagnostics, we do not keep it.

CANDIDATES = [
    ("BASE_V6_REPRODUCTION", tuple(), None),
    ("BASE_PLUS_EXPERIENCE_BINS", ("experience_bins",), None),
    ("BASE_PLUS_MISSINGNESS", ("missingness",), None),
    ("BASE_PLUS_TECH_FAMILIES", ("tech_families",), None),
    ("BASE_PLUS_TECH_AND_INTERACTIONS", ("tech_families", "interactions"), None),
    ("BASE_PLUS_POLY_SMALL", ("missingness", "tech_families", "poly_small"), None),
    ("BASE_LEAN_COMBO", ("experience_bins", "missingness", "tech_families", "interactions", "poly_small"), None),
    # Optional LASSO feature selection variants on the combined feature set.
    ("BASE_LEAN_COMBO_LASSO_TOP120", ("experience_bins", "missingness", "tech_families", "interactions", "poly_small"), 120),
    ("BASE_LEAN_COMBO_LASSO_TOP180", ("experience_bins", "missingness", "tech_families", "interactions", "poly_small"), 180),
]

all_records = []
cache = {}

for name, blocks, lasso_top_n in CANDIDATES:
    print(f"Running {name} ...")
    rec, pred_val, pred_kaggle, prepared, model = evaluate_feature_block_candidate(name, blocks, lasso_top_n)
    all_records.append(rec)
    cache[name] = {
        "pred_val": pred_val,
        "pred_kaggle": pred_kaggle,
        "prepared": prepared,
        "model": model,
        "blocks": blocks,
        "lasso_top_n": lasso_top_n,
    }
    print(f"  validation RMSE = {rec['val_rmse']:.2f}, features used = {rec['n_features_used']}")

results_df = pd.DataFrame(all_records).sort_values("val_rmse").reset_index(drop=True)
display(results_df)

results_df.to_csv(OUTPUT_DIR / "clean_feature_ablation_results.csv", index=False)

base_rmse = float(results_df.loc[results_df["candidate"] == "BASE_V6_REPRODUCTION", "val_rmse"].iloc[0])
selected_name = str(results_df.iloc[0]["candidate"])
selected_rmse = float(results_df.iloc[0]["val_rmse"])

print("\nBase v6 validation RMSE:", round(base_rmse, 2))
print("Selected candidate:", selected_name)
print("Selected validation RMSE:", round(selected_rmse, 2))

if selected_rmse > base_rmse:
    print("WARNING: none of the clean feature blocks beat the reproduced v6 baseline on validation. Keep current best submission unless Kaggle testing says otherwise.")


Running BASE_V6_REPRODUCTION ...
  validation RMSE = 36900.79, features used = 376
Running BASE_PLUS_EXPERIENCE_BINS ...
  validation RMSE = 36836.64, features used = 385
Running BASE_PLUS_MISSINGNESS ...
  validation RMSE = 36905.67, features used = 383
Running BASE_PLUS_TECH_FAMILIES ...
  validation RMSE = 36771.63, features used = 391
Running BASE_PLUS_TECH_AND_INTERACTIONS ...
  validation RMSE = 36791.04, features used = 401
Running BASE_PLUS_POLY_SMALL ...
  validation RMSE = 36937.55, features used = 464
Running BASE_LEAN_COMBO ...
  validation RMSE = 36915.57, features used = 483
Running BASE_LEAN_COMBO_LASSO_TOP120 ...
  validation RMSE = 37704.71, features used = 120
Running BASE_LEAN_COMBO_LASSO_TOP180 ...
  validation RMSE = 37382.21, features used = 180


,candidate,feature_blocks,lasso_top_n,lasso_best_params,n_features_before_selection,n_features_used,val_rmse,val_mae,fit_seconds,pred_mean,...,n_pred_gt150k,n_pred_gt200k,kaggle_pred_mean,kaggle_pred_median,kaggle_pred_p95,kaggle_pred_p99,kaggle_pred_max,kaggle_n_pred_gt100k,kaggle_n_pred_gt150k,kaggle_n_pred_gt200k
0,BASE_PLUS_TECH_FAMILIES,tech_families,none,,391,391,36771.629749,22957.080780,39.144269,47243.547374,...,1,0,46892.690657,41803.640042,97833.097709,122340.073471,179608.770892,28,1,0
1,BASE_PLUS_TECH_AND_INTERACTIONS,"tech_families,interactions",none,,401,401,36791.040535,22940.667101,39.498827,47299.894977,...,1,0,46922.355813,41324.661095,98557.700182,122749.111147,182258.752958,29,1,0
2,BASE_PLUS_EXPERIENCE_BINS,experience_bins,none,,385,385,36836.635893,22939.810973,12.962470,47431.058923,...,2,0,46892.255826,41900.342279,94780.858115,124128.398546,165555.515034,24,1,0
3,BASE_V6_REPRODUCTION,base_v6,none,,376,376,36900.793838,23053.568287,14.155177,47243.259095,...,1,0,46887.964279,41238.230142,96346.357837,122974.026278,172484.880713,27,1,0
4,BASE_PLUS_MISSINGNESS,missingness,none,,383,383,36905.666838,23058.078746,11.875516,47214.649832,...,1,0,46859.909049,41243.284160,96466.419579,123062.620739,172307.642198,27,1,0
5,BASE_LEAN_COMBO,"experience_bins,missingness,tech_families,inte...",none,,483,483,36915.567353,22746.849645,39.522159,47240.796121,...,0,0,47131.969003,41917.300160,96840.460279,117631.513177,161557.451703,24,1,0
6,BASE_PLUS_POLY_SMALL,"missingness,tech_families,poly_small",none,,464,464,36937.548025,22902.406973,39.575074,46892.361788,...,0,0,46931.804982,41546.905971,97497.309239,117489.247338,169795.137789,29,1,0
7,BASE_LEAN_COMBO_LASSO_TOP180,"experience_bins,missingness,tech_families,inte...",180,"{""alpha"": 0.03162277660168379}",483,180,37382.206055,23270.348936,85.001429,47246.729542,...,2,0,46517.825237,41699.563928,92897.409784,118440.952747,180617.967016,22,1,0
8,BASE_LEAN_COMBO_LASSO_TOP120,"experience_bins,missingness,tech_families,inte...",120,"{""alpha"": 0.03162277660168379}",483,120,37704.710124,23332.323218,88.776958,47001.817925,...,2,0,45849.883395,42716.751499,91774.666254,115680.136944,167795.112239,14,1,0



Base v6 validation RMSE: 36900.79
Selected candidate: BASE_PLUS_TECH_FAMILIES
Selected validation RMSE: 36771.63


In [10]:

# ===============================================================
# 9. Internal-test and bucket diagnostics for the selected candidate
# ===============================================================
selected = cache[selected_name]
selected_blocks = selected["blocks"]
selected_lasso_top_n = selected["lasso_top_n"]

prepared_internal = prepare_features(
    train_df_fit=df_train,
    val_df_eval=df_internal_test,
    kaggle_df_optional=df_kaggle,
    feature_blocks=selected_blocks,
)

model_internal, pred_internal, pred_kaggle_selected, selected_cols_internal, lasso_params_internal = fit_predict_svr(
    prepared_internal,
    lasso_top_n=selected_lasso_top_n
)

internal_rmse = rmse(y_internal_test_raw, pred_internal)
internal_mae = mean_absolute_error(y_internal_test_raw, pred_internal)

print("Selected candidate:", selected_name)
print("Internal-test RMSE:", round(internal_rmse, 2))
print("Internal-test MAE:", round(internal_mae, 2))
print("Selected feature count:", len(selected_cols_internal))

bucket_internal = salary_bucket_summary(y_internal_test_raw, pred_internal)
display(bucket_internal)
bucket_internal.to_csv(OUTPUT_DIR / "clean_selected_internal_bucket_diagnostics.csv", index=False)

# Compare prediction distribution to current best Kaggle submission, if available.
best_submission_path = Path("submission_weighted_tail_selected_refit_train_val.csv")
if best_submission_path.exists():
    current_best_submission = pd.read_csv(best_submission_path)
    print("\nCurrent best Kaggle prediction distribution:")
    display(current_best_submission[TARGET].describe().to_frame().T)
    print("\nSelected model Kaggle prediction distribution:")
    display(pd.Series(pred_kaggle_selected, name=TARGET).describe().to_frame().T)

    comparison_dist = pd.DataFrame([
        {"source": "current_best_submission", **prediction_distribution(current_best_submission[TARGET].to_numpy())},
        {"source": "selected_clean_model", **prediction_distribution(pred_kaggle_selected)},
    ])
    display(comparison_dist)
    comparison_dist.to_csv(OUTPUT_DIR / "kaggle_prediction_distribution_comparison.csv", index=False)


Selected candidate: BASE_PLUS_TECH_FAMILIES
Internal-test RMSE: 50522.02
Internal-test MAE: 23455.41
Selected feature count: 391


,bucket,n,mean_actual,mean_pred,rmse,mae,mse_sum,mse_contribution_pct
0,<500,12,249.250000,29619.690742,33259.752018,29370.440742,1.327453e+10,1.379483
1,500-10k,58,2718.844828,39389.133883,42078.314620,36670.289055,1.026939e+11,10.671904
2,10k-30k,73,21180.684932,28393.412924,15749.015291,10753.229473,1.810630e+10,1.881598
3,30k-60k,137,45077.379562,48630.591666,21069.779984,15822.612106,6.081918e+10,6.320302
4,60k-100k,74,74173.378378,62687.083113,23745.324328,19262.013776,4.172419e+10,4.335959
5,100k-200k,21,132373.000000,83854.208809,58643.610452,52996.121875,7.222053e+10,7.505125
6,200k+,2,557293.000000,121105.367616,571596.110583,436187.632384,6.534442e+11,67.905628


In [11]:

# ===============================================================
# 10. Small ML1 model comparison on the selected feature set
# ===============================================================
# This keeps the project documentation complete without turning the notebook into a huge search.
# All models are from the ML1 course family: Ridge, LASSO, Elastic Net, KNN, SVR.

prepared = prepare_features(
    train_df_fit=df_train,
    val_df_eval=df_val,
    kaggle_df_optional=None,
    feature_blocks=selected_blocks,
)
X_train = prepared["X_train"]
X_val = prepared["X_val"]
y_train_scaled = prepared["y_train_scaled"]
target_tf = prepared["target_tf"]
weights = make_sample_weights(prepared["y_train_raw_used"], BASE_WEIGHT_SCHEME)

models = {
    "Ridge": Ridge(alpha=10.0),
    "LASSO": Lasso(alpha=0.001, max_iter=20000, random_state=RANDOM_STATE),
    "ElasticNet": ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=20000, random_state=RANDOM_STATE),
    "KNN": KNeighborsRegressor(n_neighbors=15, weights="distance"),
    "SVR_RBF_v6_params": SVR(kernel="rbf", cache_size=1000, **BASE_SVR_PARAMS),
}

model_records = []
for model_name, model in models.items():
    print("Fitting", model_name)
    if model_name in ["KNN"]:
        model.fit(X_train, y_train_scaled)
    else:
        try:
            model.fit(X_train, y_train_scaled, sample_weight=weights)
        except TypeError:
            model.fit(X_train, y_train_scaled)

    pred_val_scaled = model.predict(X_val)
    pred_val_usd = target_tf.inverse_transform_to_usd(pred_val_scaled)

    model_records.append({
        "model": model_name,
        "val_rmse": rmse(y_val_raw, pred_val_usd),
        "val_mae": float(mean_absolute_error(y_val_raw, pred_val_usd)),
        **prediction_distribution(pred_val_usd),
    })

model_compare_df = pd.DataFrame(model_records).sort_values("val_rmse").reset_index(drop=True)
display(model_compare_df)
model_compare_df.to_csv(OUTPUT_DIR / "clean_ml1_model_comparison.csv", index=False)


Fitting Ridge
Fitting LASSO
Fitting ElasticNet
Fitting KNN
Fitting SVR_RBF_v6_params


,model,val_rmse,val_mae,pred_mean,pred_median,pred_p95,pred_p99,pred_max,n_pred_gt100k,n_pred_gt150k,n_pred_gt200k
0,SVR_RBF_v6_params,36771.629749,22957.080780,47243.547374,41324.022212,97978.041615,131890.983338,162970.473606,16,1,0
1,KNN,42414.636234,27086.084369,31638.216695,28043.091861,64281.168287,81545.421708,105261.968438,1,0,0
2,LASSO,53628.487037,32766.307650,44618.468728,27433.407253,138717.815294,250135.023495,417991.053199,34,19,9
3,ElasticNet,55602.092048,33476.990026,45145.082229,27360.780903,134204.063765,263413.155216,423990.977964,34,16,8
4,Ridge,57038.848672,34064.119656,45856.498301,27508.783007,135692.914257,278065.413213,412964.496115,35,16,8


In [12]:

# ===============================================================
# 11. Refit selected model on train + validation and create submission
# ===============================================================
# Validation and internal-test targets stayed raw throughout evaluation.
# The final model is refit on train + validation only after candidate choice.

df_trainval_refit = pd.concat([df_train, df_val], axis=0).reset_index(drop=True)

prepared_final = prepare_features(
    train_df_fit=df_trainval_refit,
    val_df_eval=df_internal_test,      # placeholder for internal diagnostics
    kaggle_df_optional=df_kaggle,
    feature_blocks=selected_blocks,
)

final_model, pred_internal_refit, pred_kaggle_final, selected_cols_final, lasso_params_final = fit_predict_svr(
    prepared_final,
    lasso_top_n=selected_lasso_top_n
)

submission = pd.DataFrame({
    ID_COL: df_kaggle[ID_COL].values,
    TARGET: pred_kaggle_final,
})

submission_path = OUTPUT_DIR / "submission_clean_v6_plus_selected_features.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission:", submission_path.resolve())
print("Final Kaggle prediction distribution:")
display(submission[TARGET].describe().to_frame().T)

summary = {
    "selected_candidate": selected_name,
    "selected_blocks": list(selected_blocks),
    "selected_lasso_top_n": selected_lasso_top_n,
    "base_validation_rmse": base_rmse,
    "selected_validation_rmse": selected_rmse,
    "selected_internal_test_rmse_before_refit": internal_rmse,
    "n_features_final": len(selected_cols_final),
    "submission_path": str(submission_path),
    "important_note": "Submit this only if diagnostics are at least as good as the current v6 benchmark; otherwise keep the current best public submission.",
}
with open(OUTPUT_DIR / "clean_v6_selected_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


Saved submission: C:\Users\natal\OneDrive\Pulpit\DSBA - UW\Semestr 2\Machine Learning\ml_classification_regression\ml-1-2026-task-2-developer-salary-prediction-regression\clean_v6_feature_outputs\submission_clean_v6_plus_selected_features.csv
Final Kaggle prediction distribution:


,count,mean,std,min,25%,50%,75%,max
annual.pay.usd,628.0,46401.72468,26105.896459,6502.180358,26650.931722,41530.599846,60645.961098,178446.18428


{
  "selected_candidate": "BASE_PLUS_TECH_FAMILIES",
  "selected_blocks": [
    "tech_families"
  ],
  "selected_lasso_top_n": null,
  "base_validation_rmse": 36900.793837824145,
  "selected_validation_rmse": 36771.6297492922,
  "selected_internal_test_rmse_before_refit": 50522.01951951215,
  "n_features_final": 391,
  "submission_path": "clean_v6_feature_outputs\\submission_clean_v6_plus_selected_features.csv",
  "important_note": "Submit this only if diagnostics are at least as good as the current v6 benchmark; otherwise keep the current best public submission."
}


## How to interpret the results

Use `BASE_V6_REPRODUCTION` as the benchmark. A new feature block is only useful if it improves validation RMSE without making the internal-test and high-tail bucket diagnostics clearly worse.

The generated submission is saved as:

`clean_v6_feature_outputs/submission_clean_v6_plus_selected_features.csv`

If the selected model does not beat the v6 baseline, keep the current best Kaggle submission instead.
